## Primer Paso (NSL-KDD)

### Pre-Algorithm


In [ ]:
!pip install numpy==1.26.4 scipy==1.14.1 pandas==2.2.2 joblib==1.4.2 threadpoolctl==3.6.0 matplotlib==3.9.3
!pip install -q scikit-learn==1.6.1 lightgbm==4.5.0 catboost==1.2.7
!pip install -q scapy==2.5.0
!pip install -q tensorflow==2.18.0
!pip install -q optuna==4.2.1
!pip install -q shap==0.46.0 lime==0.2.0.1
!pip install -q deap==1.4.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 122.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 98.7/98.7 MB 185.6 MB/s eta 0:00:01^C


### Libraries

In [ ]:
# =========================================
# NSL-KDD en Google Colab
# =========================================

import itertools
import time
import warnings
import os # accessing directory structure

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from pandas.api.types import is_numeric_dtype

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    auc,
    roc_curve,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.model_selection import cross_val_score, cross_val_predict, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier

SEED_FIRST_TRAIN = 2
SEED_SECOND_TRAIN = 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
for dirname, _, filenames in os.walk('/content/drive/MyDrive/nsl-kdd'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train=pd.read_csv('/content/drive/MyDrive/nsl-kdd/KDDTrain+.txt')
# test=pd.read_csv('/content/drive/MyDrive/nsl-kdd/KDDTest+.txt')
### https://www.kaggle.com/code/eneskosar19/intrusion-detection-system-nsl-kdd

### ADJUST COLUMNS

In [ ]:
train.columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
'num_access_files', 'num_outbound_cmds', 'is_host_login',
'is_guest_login', 'count', 'srv_count', 'serror_rate',
'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
'dst_host_srv_count', 'dst_host_same_srv_rate','dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
'dst_host_srv_rerror_rate', 'attack','level']

### DATA CLEANING

In [ ]:
train.isnull().sum()

In [ ]:
def unique_values(train, columns):
    for column_name in columns:
        print(f"Column: {column_name}\n{'-'*30}")
        unique_vals = train[column_name].unique()
        value_counts = train[column_name].value_counts()
        print(f"Unique Values ({len(unique_vals)}): {unique_vals}\n")
        print(f"Value Counts:\n{value_counts}\n{'='*40}\n")

In [ ]:
cat_features = train.select_dtypes(include='object').columns
unique_values(train, cat_features)

### DUPLICATES

In [ ]:
train.duplicated().sum()

### CLASSIFY ATTACK OR NOT

In [ ]:
attack_n = []
for i in train.attack :
  if i == 'normal':
    attack_n.append("normal")
  else:
    attack_n.append("attack")
train['attack'] = attack_n

In [ ]:
train['attack'].unique()

### PREPROCESSING (obsolete)

In [ ]:
#### ENCODING
cat_features = train.select_dtypes(include='object').columns
cat_features


In [ ]:
from sklearn import preprocessing
le=preprocessing.LabelEncoder()
clm=['protocol_type', 'service', 'flag', 'attack']
for x in clm:
    train[x]=le.fit_transform(train[x])

In [ ]:
#### TRAIN-TEST-SPLIT
from sklearn.model_selection import train_test_split

X = train.drop(["attack"], axis=1)
y = train["attack"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.1,random_state=43)
train_index = X_train.columns
train_index

In [ ]:
####  Feature Engineering
from sklearn.feature_selection import mutual_info_classif
mutual_info = mutual_info_classif(X_train, y_train)
mutual_info = pd.Series(mutual_info)
mutual_info.index = train_index
mutual_info.sort_values(ascending=False)
mutual_info.sort_values(ascending=False).plot.bar(figsize=(20, 5));

In [ ]:
#### Feature Selection
from sklearn.feature_selection import SelectKBest
Select_features = SelectKBest(mutual_info_classif, k=30)
Select_features.fit(X_train, y_train)
train_index[Select_features.get_support()]

columns=['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'wrong_fragment', 'hot', 'logged_in', 'num_compromised',
       'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate']

X_train=X_train[columns]
X_test=X_test[columns]

In [ ]:
#### Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Combinar X_test y y_test en un solo DataFrame
test_data = pd.DataFrame(X_test, columns=columns)  # Usamos las mismas columnas seleccionadas
test_data['attack'] = y_test.reset_index(drop=True)  # Agregamos la columna de etiquetas

# Guardar los datos de prueba en un archivo JSON
ruta_test_json = '/content/drive/MyDrive/nsl-kdd/test_data_cont.json'
test_data.to_json(ruta_test_json, orient='records', lines=True)

# Guardar en CSV
ruta_test_csv = '/content/drive/MyDrive/nsl-kdd/test_data_cont.csv'
test_data.to_csv(ruta_test_csv, index=False)

print(f"Datos de prueba guardados en JSON: {ruta_test_json}")
print(f"Datos de prueba guardados en CSV: {ruta_test_csv}")


### PREPROCESSING (NEW) - NO USADO

In [ ]:
# import pandas as pd
# import os
# from sklearn import preprocessing
# from sklearn.model_selection import train_test_split
# from sklearn.feature_selection import mutual_info_classif, SelectKBest
# from sklearn.preprocessing import StandardScaler

# # === 1. ENCODING ===
# cat_features = train.select_dtypes(include='object').columns
# le = preprocessing.LabelEncoder()

# # Columnas a codificar
# clm = ['protocol_type', 'service', 'flag', 'attack']
# for col in clm:
#     train[col] = le.fit_transform(train[col])

# # === 2. TRAIN-TEST SPLIT ===
# X = train.drop(["attack"], axis=1)
# y = train["attack"]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=43)
# train_index = X_train.columns

# # === 3. FEATURE ENGINEERING ===
# mutual_info = pd.Series(mutual_info_classif(X_train, y_train), index=train_index)
# mutual_info.sort_values(ascending=False).plot.bar(figsize=(20, 5))

# # === 4. FEATURE SELECTION ===
# Select_features = SelectKBest(mutual_info_classif, k=30)
# Select_features.fit(X_train, y_train)

# # Seleccionar las mejores 15 features manualmente
# columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
#            'dst_bytes', 'wrong_fragment', 'hot', 'logged_in', 'num_compromised',
#            'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate']

# X_train = X_train[columns]
# X_test = X_test[columns]

# # === 5. SCALING  ===
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # === 6. GUARDAR TEST_CONT_DATA  ===
# # OPCIONAL: Guardar versión no escalada del test set

# output_path = '/content/drive/MyDrive/nsl-kdd'
# os.makedirs(output_path, exist_ok=True)

# test_data = X_test.copy()
# test_data['attack'] = y_test.reset_index(drop=True)

# ruta_raw_csv = os.path.join(output_path, "test_data_cont.csv")
# ruta_raw_json = os.path.join(output_path, "test_data_cont.json")

# test_data.to_csv(ruta_raw_csv, index=False)
# test_data.to_json(ruta_raw_json, orient='records', lines=True)

# print(f"Versión no escalada guardada como (raw):")
# print(f" - CSV: {ruta_raw_csv}")
# print(f" - JSON: {ruta_raw_json}")


# # Reconstruir DataFrames escalados con nombres de columnas
# X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=columns, index=X_train.index)
# X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=columns, index=X_test.index)

# # Agregar etiquetas
# X_train_scaled_df['attack'] = y_train.reset_index(drop=True)
# X_test_scaled_df['attack'] = y_test.reset_index(drop=True)

# # === 7. GUARDADO EN CSV Y JSON ===
# # Guardar conjunto de prueba combinado
# X_test_scaled_df.to_csv(os.path.join(output_path, "X_test_scaled.csv"), index=False)
# # X_test_scaled_df.to_json(os.path.join(output_path, "test_data_cont.json"), orient='records', lines=True)

# # guardar entrenamiento:
# X_train_scaled_df.to_csv(os.path.join(output_path, "X_train_scaled.csv"), index=False)

# print(f"Datos escalados guardados en {output_path}")


## Segundo Paso (UNSW15)

### PRE-ALGORITHM

#### 1 SECTION

In [ ]:
# Instalación de bibliotecas faltantes xd
!pip install xgboost==2.1.3
!pip install tensorflow==2.18.0

### CODE

In [ ]:
# =========================================
# Código Final para UNSW-15 en Google Colab
# =========================================

import os  # Acceso a directorios
import numpy as np
import pandas as pd
import missingno

# Machine learning
from sklearn.model_selection import train_test_split

# Definir semilla para reproducibilidad
SEED_FIRST_TRAIN = 57

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verificar archivos en la carpeta de Drive
import os
for dirname, _, filenames in os.walk('/content/drive/MyDrive/UNSW_NB15/UNSW_NB15'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Ajustar rutas de los archivos de entrenamiento y prueba
train = pd.read_csv('/content/drive/MyDrive/UNSW_NB15/UNSW_NB15_training-set.csv')
test = pd.read_csv('/content/drive/MyDrive/UNSW_NB15/UNSW_NB15_testing-set.csv')

# Unir los conjuntos de entrenamiento y prueba
data = pd.concat([train, test]).reset_index(drop=True)

# Identificar columnas categóricas y numéricas
cols_cat = data.select_dtypes('object').columns
cols_numeric = data.select_dtypes(include=[np.number]).columns

# Manejo de valores faltantes o incorrectos
data['service'] = np.where(data['service'] == '-', 'None', data['service'])

# Automatizar limpieza de valores incorrectos
def clean_data(data, cols):
    for col in cols:
        data[col] = np.where(data[col] == '-', 'None', data[col])
    return data

data = clean_data(data, data.columns)

# Eliminar columnas innecesarias
data = data.drop(['id', 'attack_cat'], axis=1)
cols_cat = cols_cat.drop(['attack_cat'])

# Aplicar One-Hot Encoding a variables categóricas
data_encoded = pd.get_dummies(data, columns=cols_cat)

# Normalizar columnas numéricas con la normalización original
cols_numeric = list(cols_numeric)
cols_numeric.remove('label')
cols_numeric.remove('id')
data_encoded[cols_numeric] = data_encoded[cols_numeric].astype('float')
data_encoded[cols_numeric] = (data_encoded[cols_numeric] - np.min(data_encoded[cols_numeric])) / np.std(data_encoded[cols_numeric])

# Separar características (X) y etiquetas (Y)
X = data_encoded.drop('label', axis=1)
Y = data_encoded['label']

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=SEED_FIRST_TRAIN)

# Convertir etiquetas a enteros
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# ============================
# Guardar en Google Drive
# ============================

# Crear carpeta en Google Drive si no existe
ruta_drive = '/content/drive/MyDrive/UNSW_NB15/UNSW_NB15'
os.makedirs(ruta_drive, exist_ok=True)

# Guardar el conjunto de prueba en CSV
X_test_copy = X_test.copy()
X_test_copy['attack'] = y_test.reset_index(drop=True)  # Agregar columna de etiquetas

ruta_csv = os.path.join(ruta_drive, "test_data_cont_unsw15.csv")
X_test_copy.to_csv(ruta_csv, index=False)

# caching
print(f"[✔] Datos de prueba guardados en: {ruta_csv}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


[✔] Datos de prueba guardados en: /content/drive/MyDrive/UNSW_NB15/UNSW_NB15/test_data_cont_unsw15.csv


## Tercer paso Paso (CICIDS2018)

## Analisis de Resultados

### Install

#### Install library

In [ ]:
# Instalación de bibliotecas
!pip install scapy==2.5.0
!pip install joblib==1.4.2
!pip install scikit-learn==1.6.1
!pip install numpy==1.26.4
!pip install pandas==2.2.2
!pip install matplotlib==3.9.3
!pip install catboost==1.2.7
!pip install xgboost==2.1.3
!pip install lightgbm==4.5.0
!pip install tensorflow==2.18.0
!pip install seaborn==0.13.2
!pip install deap==1.4.2
!pip install missingno==0.5.2
!pip install optuna==4.2.1
!pip install shap==0.46.0
!pip install lime==0.2.0.1
!pip install scikeras==0.13.0
!pip install imbalanced-learn==0.12.4
!pip install -q \
  ml-dtypes==0.4.1 \
  jax==0.4.29 \
  jaxlib==0.4.29


  Using cached tensorflow-2.18.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached tensorflow-2.18.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (615.5 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.18.0 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.18.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.18.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 10.7 MB/s eta 0:00:00
  Attempting uninstall: shap
    Found e

In [ ]:
!pip uninstall -y optax orbax-checkpoint flax


Found existing installation: optax 0.2.6
Uninstalling optax-0.2.6:
  Successfully uninstalled optax-0.2.6
Found existing installation: orbax-checkpoint 0.11.28
Uninstalling orbax-checkpoint-0.11.28:
  Successfully uninstalled orbax-checkpoint-0.11.28
Found existing installation: flax 0.10.7
Uninstalling flax-0.10.7:
  Successfully uninstalled flax-0.10.7


#### Bibliotecas

In [ ]:
# =========================================
# Código Final para CICIDS2018 en Google Colab
# =========================================

# 0. Preparación del Entorno
from google.colab import drive
drive.mount('/content/drive')

# Importar Librerías Necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

# Librerías para Modelos y Evaluación
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold)
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_curve, auc)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, plot_importance
from sklearn.feature_selection import SelectFromModel
from sklearn.utils import class_weight
from sklearn.impute import SimpleImputer
import joblib
import json
import time
import optuna

from joblib import dump, load

from tensorflow.keras import Model as KerasModel
from tensorflow.keras.models import load_model  # Para cargar .keras

# Librerías para Manejar el Desbalanceo
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline  # <--- Usamos Pipeline de imblearn sin renombrar

# LightGBM
import lightgbm as lgb

# Otros Modelos
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier

#  TF
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
# from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from scikeras.wrappers import KerasClassifier


# DEAP (Algoritmos Genéticos)
from deap import base, creator, tools, algorithms

# Librerías para Medir el Tiempo de Ejecución
# %load_ext autoreload
# %autoreload 2

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Code

In [ ]:
import os
import joblib
import traceback
import numpy as np
import pandas as pd

# === 1) RECONSTRUIR EL WRAPPER PARA LGBM DE NSLKDD ===
class LGBMBoosterWrapper:
    """
    Envoltorio sklearn-like para el Booster de LightGBM.
      - self.classes_ = np.array([0, 1]) para que CalibratedClassifierCV
        no falle al hacer check_is_fitted(..., ["classes_"]).
    """
    def __init__(self, booster, feature_names):
        self.booster = booster
        # nombres de columnas que usó el modelo al entrenar
        self.feature_names_in_ = np.array(feature_names)
        # para que sklearn piense que está “fiteado”
        self.classes_ = np.array([0, 1])

    def _to_numpy(self, X):
        """
        Caso general:
          - Si las columnas de X coinciden con feature_names_in_ -> reindexar por nombre.
          - Si NO coinciden (caso NSL-KDD viejo: feature_0 vs duration, ...) -> usar el orden actual,
            sin reindexar, para no meter todo ceros.
        """
        if isinstance(X, pd.DataFrame):
            expected = list(self.feature_names_in_)
            cols = list(X.columns)

            # Caso bueno: los nombres coinciden o están contenidos
            if set(expected).issubset(cols):
                X = X.reindex(columns=expected, fill_value=0.0)
                return X.to_numpy(dtype=np.float32)

            # Caso no tan bueno: nombres distintos -> no reindexo, uso el orden actual
            return X.to_numpy(dtype=np.float32)

        # Si no es DataFrame, solo convierto a numpy
        return np.asarray(X, dtype=np.float32)

    def predict_proba(self, X):
        X_np = self._to_numpy(X)
        # En binario, booster.predict devuelve prob de clase positiva
        p = self.booster.predict(X_np)
        if p.ndim == 1:
            p = np.clip(p, 1e-7, 1 - 1e-7)
            return np.vstack([1 - p, p]).T
        if p.shape[1] == 1:
            pp = np.clip(p[:, 0], 1e-7, 1 - 1e-7)
            return np.vstack([1 - pp, pp]).T
        return p

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba[:, 1] >= 0.5).astype(int)


# === 2) EL LOADER / ===
try:
    from tensorflow.keras.models import load_model as keras_load_model
except ImportError:
    keras_load_model = None

def load_any_model(filepath):
    """
    Carga un modelo .pkl (joblib) o .keras (Keras).
    Devuelve el objeto modelo o None si falla.
    """
    ext = os.path.splitext(filepath)[1].lower()
    try:
        if ext == ".pkl":
            m = joblib.load(filepath)   # aquí ya existe LGBMBoosterWrapper en __main__
        elif ext == ".keras":
            if keras_load_model is None:
                raise ImportError("TensorFlow no está instalado.")
            m = keras_load_model(filepath, compile=False)
        else:
            print(f"La extensión no es soportada para {ext} en {filepath}")
            return None
        print(f"Se cargo: {filepath}")
        return m
    except Exception as e:
        print(f"Error cargando {filepath}: {type(e).__name__}: {e}")
        traceback.print_exc(limit=1)
        return None

def discover_and_load_models(base_dir):
    """
    Recorre cada subcarpeta de base_dir y carga modelos en:
      .pkl, .keras
    Imprime al entrar en cada carpeta de dataset.
    Retorna un dict { model_name: model_obj }.
    """
    models = {}
    for dataset_folder in os.listdir(base_dir):
        ds_path = os.path.join(base_dir, dataset_folder)
        if not os.path.isdir(ds_path):
            continue

        print(f"\n Carpeta Actual [{dataset_folder}]")

        prefix = dataset_folder.lower().replace('-', '').replace(' ', '_')
        for root, _, files in os.walk(ds_path):
            for fn in files:
                ext = os.path.splitext(fn)[1].lower()
                if ext in (".pkl", ".keras"):
                    full_path = os.path.join(root, fn)
                    key = f"{prefix}_{os.path.splitext(fn)[0]}".lower().replace(' ', '_')
                    model = load_any_model(full_path)
                    if model is not None:
                        models[key] = model

    print(f"\n Total de modelos cargados: {len(models)}")
    return models

# === USO ===
base_dir = "/content/drive/MyDrive/Modelos"
if not os.path.exists(base_dir):
    raise FileNotFoundError(f"No existe el directorio: {base_dir}")

models = discover_and_load_models(base_dir)



 Carpeta Actual [nslkdd]
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/XGB_model.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/svc_best_clf_model.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/gnb_model.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/logistic_regression_model.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/catboost_model.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/simple_neural_network_model.keras
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/decision_tree_model.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/rf_stacking_model_optimized.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/lightgbm_model.pkl
Se cargo: /content/drive/MyDrive/Modelos/nslkdd/knn_model.pkl

 Carpeta Actual [unsw15]
Se cargo: /content/drive/MyDrive/Modelos/unsw15/gnb_model_unsw.pkl
Se cargo: /content/drive/MyDrive/Modelos/unsw15/decision_tree_model_unsw.pkl
Se cargo: /content/drive/MyDrive/Modelos/unsw15/logistic_regression_model_unsw.pkl
Se cargo: /content

In [ ]:
import pandas as pd

# === RUTAS DE LOS DATASETS ===
csv_paths = {
    'nslkdd': '/content/drive/MyDrive/nsl-kdd/test_data_cont.csv',
    'unsw15': '/content/drive/MyDrive/UNSW_NB15/UNSW_NB15/test_data_cont_unsw15.csv',
    'cicids2018': '/content/drive/MyDrive/CICIDS2018/df_all_preprocessed.csv'
}

# === CARGA DE LOS DATASETS ===
datasets = {}

for name, path in csv_paths.items():
    try:
        df = pd.read_csv(path)
        datasets[name] = df
        print(f" {name} cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    except Exception as e:
        print(f" Error al cargar {name}: {e}")

# === VERIFICAR QUE TODO SE CARGÓ ===
print("\n Datasets cargados:")
for name, df in datasets.items():
    print(f"- {name}: {df.shape}")


 nslkdd cargado: 12598 filas, 16 columnas
 unsw15 cargado: 77302 filas, 197 columnas
 cicids2018 cargado: 1352924 filas, 38 columnas

 Datasets cargados:
- nslkdd: (12598, 16)
- unsw15: (77302, 197)
- cicids2018: (1352924, 38)


In [ ]:
# PARA EVALUAR EL KERAS
def keras_pred_and_proba_ids(model, X):
    """
    - X -> DataFrame numérico
    - model.predict -> probabilidades
    - Normaliza a [p0, p1]
    - Etiquetas binarias con umbral 0.5 sobre p1
    """
    # Aseguramos DataFrame
    X_df = pd.DataFrame(X)
    # A float32
    arr = X_df.to_numpy(dtype=np.float32)

    # Predicciones Keras
    probas = model.predict(arr, verbose=0)
    probas = np.asarray(probas)

    # Normalizar a dos columnas [p0, p1]
    if probas.ndim == 2 and probas.shape[1] == 1:
        # salida [n,1] -> prob clase 1
        probas = np.hstack([1.0 - probas, probas])
    elif probas.ndim == 1:
        # salida [n] -> prob clase 1
        p1 = probas.reshape(-1, 1)
        probas = np.hstack([1.0 - p1, p1])
    # si ya es (n,2), se deja tal cual

    # Etiquetas binarias
    preds = (probas[:, 1] >= 0.5).astype(int)
    return preds, probas


#### Tratamiento final para cicids2018


In [ ]:
from sklearn.model_selection import train_test_split

# =======================
# 6. Preparación de Datos para Entrenamiento
# =======================
print("\n[INFO] Preparando datos para entrenamiento (binario)")

# Usar el dataset cicids2018
df_all = datasets["cicids2018"]

# a) Separar características y etiquetas (binario)
X_binary = df_all.drop(['Label', 'Threat', 'Attack Type'], axis=1)
y_binary = df_all['Threat']

# Convertir etiquetas a índices numéricos (solo si fuera necesario,
# pero ya en el 'Threat' tenemos el 0/1 XD)
binary_label_mapping = {0: 0, 1: 1}
y_binary_indices = y_binary.map(binary_label_mapping)

# b) Separar características y etiquetas (multiclase) al final no se hizo multi
X_multi = df_all.drop(['Label', 'Threat', 'Attack Type'], axis=1)
y_multi = df_all['Attack Type']

# Crear un mapeo a índices numéricos
unique_multi_labels = y_multi.unique()
multi_label_mapping = {label: idx for idx, label in enumerate(unique_multi_labels)}
y_multi_indices = y_multi.map(multi_label_mapping)

print(f" - X_binary shape: {X_binary.shape}, y_binary_indices shape: {y_binary_indices.shape}")
print(f" - X_multi shape: {X_multi.shape}, y_multi_indices shape: {y_multi_indices.shape}")


# =======================
# 7. División en Conjuntos de Entrenamiento y Prueba (Antes de SMOTE)
# =======================
print("\n[INFO] Dividiendo datos en entrenamiento y prueba...")

# Binario
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_binary,
    y_binary_indices,
    test_size=0.2,
    random_state=42,
    stratify=y_binary_indices
)

# Multiclase
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi,
    y_multi_indices,
    test_size=0.2,
    random_state=42,
    stratify=y_multi_indices
)

print(" - Conjuntos binarios:")
print(f"   X_train_bin: {X_train_bin.shape}, y_train_bin: {y_train_bin.shape}")
print(f"   X_test_bin: {X_test_bin.shape}, y_test_bin: {y_test_bin.shape}")

print(" - Conjuntos multicategoría:")
print(f"   X_train_multi: {X_train_multi.shape}, y_train_multi: {y_train_multi.shape}")
print(f"   X_test_multi: {X_test_multi.shape}, y_test_multi: {y_test_multi.shape}")

# # Asegurar que sean int
y_train_bin = y_train_bin.astype(int)
y_test_bin = y_test_bin.astype(int)


[INFO] Preparando datos para entrenamiento (binario)
 - X_binary shape: (1352924, 35), y_binary_indices shape: (1352924,)
 - X_multi shape: (1352924, 35), y_multi_indices shape: (1352924,)

[INFO] Dividiendo datos en entrenamiento y prueba...
 - Conjuntos binarios:
   X_train_bin: (1082339, 35), y_train_bin: (1082339,)
   X_test_bin: (270585, 35), y_test_bin: (270585,)
 - Conjuntos multicategoría:
   X_train_multi: (1082339, 35), y_train_multi: (1082339,)
   X_test_multi: (270585, 35), y_test_multi: (270585,)


In [ ]:
print("Evaluando modelos para Clasificación Binaria...")
binary_class_labels = ['Benign', 'Malicious']

Evaluando modelos para Clasificación Binaria...


In [ ]:
print("\n Variables de datasets disponibles:")
for name in datasets:
    print(f"- {name}: {datasets[name].shape}")



 Variables de datasets disponibles:
- nslkdd: (12598, 16)
- unsw15: (77302, 197)
- cicids2018: (1352924, 38)


In [ ]:
import pandas as pd
import os

# Ruta donde se guardo los archivos
output_path = '/content/drive/MyDrive/nsl-kdd'

# === Cargar los conjuntos escalados ===
X_test_scaled_df = pd.read_csv(os.path.join(output_path, "X_test_scaled.csv"))
X_train_scaled_df = pd.read_csv(os.path.join(output_path, "X_train_scaled.csv"))

# Verificar las dimensiones
print(" Archivos cargados:")
print(f" - X_train_scaled_df: {X_train_scaled_df.shape}")
print(f" - X_test_scaled_df: {X_test_scaled_df.shape}")

# Ver primeras filas
print("\n Primeras filas del test set:")
display(X_test_scaled_df.head())

 Archivos cargados:
 - X_train_scaled_df: (113374, 16)
 - X_test_scaled_df: (12598, 16)

 Primeras filas del test set:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,wrong_fragment,hot,logged_in,num_compromised,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,attack
0,-0.110148,-0.125021,-0.442381,0.751851,-0.007871,-0.005045,-0.089501,-0.094155,1.237773,-0.011348,-0.604246,-0.176124,-0.637510,-0.63225,-0.374948,NaN
1,-0.110148,-0.125021,-0.442381,-2.220293,-0.007919,-0.005089,-0.089501,-0.094155,-0.807902,-0.011348,-0.726329,-0.368136,-0.637510,-0.63225,2.741791,NaN
2,-0.110148,-0.125021,1.086704,-2.220293,-0.007919,-0.005089,-0.089501,-0.094155,-0.807902,-0.011348,-0.098473,-0.368136,-0.413589,-0.63225,2.430118,NaN
3,-0.110148,-0.125021,-0.442381,0.751851,-0.007882,-0.004967,-0.089501,-0.094155,1.237773,-0.011348,-0.560645,-0.066403,-0.637510,-0.63225,-0.374948,NaN
4,-0.110148,-2.467470,-1.054015,0.751851,-0.007918,-0.005089,-0.089501,-0.094155,-0.807902,-0.011348,-0.726329,0.290191,-0.637510,-0.63225,-0.374948,NaN


#### NSLKDD

In [ ]:
from sklearn.metrics            import f1_score, accuracy_score, classification_report
import pandas as pd
import numpy as np
from IPython.display           import display
from sklearn.metrics import f1_score, accuracy_score, classification_report

# ——————————————
# 1) Prepara los datos
# ——————————————
# 1) Datos de prueba
X_test_scaled_df = datasets["nslkdd"]
X_test_final = X_test_scaled_df.drop("attack", axis=1)
y_test_final = X_test_scaled_df["attack"]

# Elimina filas con etiqueta NaN
# mask = y_test_final.notna()
# X_test_final = X_test_final.loc[mask]
# y_test_final = y_test_final.loc[mask]

# ——————————————————————————
# 2) Filtra solo NSL-KDD (prefijo "nslkdd_")
# ——————————————————————————
nslkdd_models = {
    name: mdl
    for name, mdl in models.items()
    if name.lower().startswith("nslkdd_")
}

# nombres de los 4 modelos base (sin prefijo):
base_keys = [
    "nslkdd_decision_tree_model",
    "nslkdd_logistic_regression_model",
    "nslkdd_svc_best_clf_model",
    "nslkdd_knn_model"
]

# ———————————————————————————————
# 3) Recorre y evalúa cada modelo
# ———————————————————————————————
for model_name, model in nslkdd_models.items():
    print(f"\n Evaluando el modelo: {model_name}")

    # —— Stacking ——
    if "rf_stacking_model_optimized" in model_name:
        # 1) preds de base
        base_preds = [
            nslkdd_models[bk].predict(X_test_final)
            for bk in base_keys
        ]
        # 2) apilar features originales + preds
        X_orig = X_test_final.values                      # (n,15)
        X_stack = np.hstack([X_orig] + [p.reshape(-1,1) for p in base_preds])  # (n,19)

        # 3) predict stacking
        y_pred = model.predict(X_stack)
        try:
            proba = model.predict_proba(X_stack)
            has_proba = True
            # para consistencia con "proba_clase_i":
            y_proba = proba
            print(" Stacking: probabilidades obtenidas.")
        except:
            has_proba = False
            y_proba = None
            print(" Stacking: sin predict_proba().")

    # ——— Modelos Keras ———
    elif isinstance(model, KerasModel):
      # Aquí usamos el helper alineado con el IDS-ML
      y_pred, y_proba = keras_pred_and_proba_ids(model, X_test_final)
      has_proba = True
      print(" Keras salida")

    # —— Modelos sklearn restantes ——
    else:
        # 1) predicción directa
        model_to_use = model

        # 2) predicción / proba
        y_pred = model_to_use.predict(X_test_final)
        try:
            y_proba = model_to_use.predict_proba(X_test_final)
            has_proba = True
            print(" Probabilidades obtenidas (modelo original).")
        except Exception:
            has_proba = False
            y_proba = None
            print(" Probabilidades NO disponibles.")


    # —— Métricas & reporte ——
    f1  = f1_score(y_test_final, y_pred, average="weighted")
    acc = accuracy_score(y_test_final, y_pred)
    print(f" F1-score (w): {f1:.4f}   Accuracy: {acc:.4f}")
    print(" Classification Report:")
    print(classification_report(y_test_final, y_pred, zero_division=0))

    # —— Muestra primeras filas ——
    df_out = pd.DataFrame({"y_true": y_test_final, "y_pred": y_pred})
    if has_proba:
        proba_df = pd.DataFrame(
            y_proba,
            columns=[f"proba_clase_{i}" for i in range(y_proba.shape[1])]
        )
        df_out = pd.concat([df_out.reset_index(drop=True), proba_df], axis=1)

    print(" Primeras predicciones:")
    display(df_out.head())




 Evaluando el modelo: nslkdd_xgb_model
 Probabilidades obtenidas (modelo original).
 F1-score (w): 0.9965   Accuracy: 0.9965
 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5828
           1       1.00      1.00      1.00      6770

    accuracy                           1.00     12598
   macro avg       1.00      1.00      1.00     12598
weighted avg       1.00      1.00      1.00     12598

 Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1,1,0.001642,0.998358
1,1,1,0.017005,0.982995
2,0,0,0.997723,0.002278
3,1,1,0.001487,0.998513
4,0,0,0.994979,0.005021



 Evaluando el modelo: nslkdd_svc_best_clf_model
 Probabilidades NO disponibles.
 F1-score (w): 0.9834   Accuracy: 0.9834
 Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      5828
           1       0.98      0.99      0.98      6770

    accuracy                           0.98     12598
   macro avg       0.98      0.98      0.98     12598
weighted avg       0.98      0.98      0.98     12598

 Primeras predicciones:


,y_true,y_pred
0,1,1
1,1,1
2,0,0
3,1,1
4,0,0



 Evaluando el modelo: nslkdd_gnb_model
 Probabilidades obtenidas (modelo original).
 F1-score (w): 0.8914   Accuracy: 0.8919
 Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.84      0.88      5828
           1       0.87      0.93      0.90      6770

    accuracy                           0.89     12598
   macro avg       0.89      0.89      0.89     12598
weighted avg       0.89      0.89      0.89     12598

 Primeras predicciones:


...

,y_true,y_pred,proba_clase_0,proba_clase_1
0,1,1,0.000000,1.000000
1,1,1,0.003299,0.996701
2,0,0,1.000000,0.000000
3,1,1,0.000000,1.000000
4,0,0,1.000000,0.000000



 Evaluando el modelo: nslkdd_lightgbm_model
 Probabilidades obtenidas (modelo original).
 F1-score (w): 0.9988   Accuracy: 0.9988
 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5828
           1       1.00      1.00      1.00      6770

    accuracy                           1.00     12598
   macro avg       1.00      1.00      1.00     12598
weighted avg       1.00      1.00      1.00     12598

 Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1,1,1.000000e-07,9.999999e-01
1,1,1,2.985753e-03,9.970142e-01
2,0,0,9.999999e-01,1.000000e-07
3,1,1,1.000000e-07,9.999999e-01
4,0,0,9.999999e-01,1.000000e-07



 Evaluando el modelo: nslkdd_knn_model
 Probabilidades obtenidas (modelo original).
 F1-score (w): 0.9980   Accuracy: 0.9980
 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5828
           1       1.00      1.00      1.00      6770

    accuracy                           1.00     12598
   macro avg       1.00      1.00      1.00     12598
weighted avg       1.00      1.00      1.00     12598

 Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1,1,0.0,1.0
1,1,1,0.0,1.0
2,0,0,1.0,0.0
3,1,1,0.0,1.0
4,0,0,1.0,0.0


#### UNSW15

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tensorflow.keras import Model as KerasModel
import pandas as pd
import numpy as np
from IPython.display import display

# 1) Datos de prueba
unsw_df = datasets["unsw15"]
X_unsw = unsw_df.drop("attack", axis=1)
y_unsw = unsw_df["attack"]

# Elimina filas con etiqueta NaN
mask = y_unsw.notna()
X_unsw = X_unsw.loc[mask]
y_unsw = y_unsw.loc[mask]

# 2) Filtrar modelos UNSW15
unsw_models = {
    name: mdl
    for name, mdl in models.items()
    if name.lower().startswith("unsw15_")
}

# 3) Los 4 modelos base para el stacking
base_keys_unsw = [
    "unsw15_decision_tree_model_unsw",
    "unsw15_logistic_regression_model_unsw",
    "unsw15_svc_model_unsw",
    "unsw15_knn_model_unsw"
]

# 4) Recorre y evalúa cada modelo
for model_name, model in unsw_models.items():
    print(f"\n Evaluando modelo: {model_name}")

    # ——— RF apilado ———
    if "rf_stacking_model_optimized_unsw" in model_name:
        # 1) preds de cada base
        base_preds = [
            unsw_models[bk].predict(X_unsw)
            for bk in base_keys_unsw
        ]
        # 2) apilar features originales + preds
        X_orig  = X_unsw.values                           # shape (n, m)
        X_stack = np.hstack([X_orig] + [p.reshape(-1,1) for p in base_preds])
        # 3) predecir con el RF apilado
        y_pred = model.predict(X_stack)
        try:
            y_proba = model.predict_proba(X_stack)
            has_proba = True
            print(" Stacking: probabilidades obtenidas.")
        except:
            y_proba = None
            has_proba = False
            print(" Stacking: sin predict_proba().")

    # ——— Modelos Keras ———
    elif isinstance(model, KerasModel):
        y_pred, y_proba = keras_pred_and_proba_ids(model, X_unsw)
        has_proba = True
        print(" Keras salida.")

    # ——— Resto de sklearn ———
    else:
        # a) predicción directa
        model_to_use = model

        # b) predicción y probas
        y_pred = model_to_use.predict(X_unsw)
        try:
            y_proba = model_to_use.predict_proba(X_unsw)
            has_proba = True
            print(" Probabilidades obtenidas (modelo original).")
        except Exception:
            has_proba = False
            y_proba = None
            print(" No fue posible obtener probabilidades.")


    # ——— Métricas & reporte ———
    f1  = f1_score(y_unsw, y_pred, average="weighted")
    acc = accuracy_score(y_unsw, y_pred)
    print(f" F1-score (weighted): {f1:.4f}   Accuracy: {acc:.4f}")
    print(" Classification Report:")
    print(classification_report(y_unsw, y_pred, zero_division=0))

    # ——— Primeras predicciones ———
    df_out = pd.DataFrame({"y_true": y_unsw, "y_pred": y_pred})
    if has_proba:
        proba_cols = [f"proba_clase_{i}" for i in range(y_proba.shape[1])]
        proba_df   = pd.DataFrame(y_proba, columns=proba_cols)
        df_out     = pd.concat([df_out.reset_index(drop=True), proba_df], axis=1)

    print(" Primeras predicciones:")
    display(df_out.head())



 Evaluando modelo: unsw15_gnb_model_unsw
 Probabilidades obtenidas (modelo original).
 F1-score (weighted): 0.4890   Accuracy: 0.4812
 Classification Report:
              precision    recall  f1-score   support

         0.0       0.36      0.57      0.44      8314
         1.0       0.64      0.43      0.52     14723

    accuracy                           0.48     23037
   macro avg       0.50      0.50      0.48     23037
weighted avg       0.54      0.48      0.49     23037

 Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1.0,1,7.645162e-12,1.000000
1,1.0,1,1.508967e-07,1.000000
2,1.0,1,3.933995e-13,1.000000
3,0.0,1,5.764015e-42,1.000000
4,1.0,0,9.998272e-01,0.000173



 Evaluando modelo: unsw15_decision_tree_model_unsw
 Probabilidades obtenidas (modelo original).
 F1-score (weighted): 0.5340   Accuracy: 0.5295
 Classification Report:
              precision    recall  f1-score   support

         0.0       0.36      0.40      0.38      8314
         1.0       0.64      0.60      0.62     14723

    accuracy                           0.53     23037
   macro avg       0.50      0.50      0.50     23037
weighted avg       0.54      0.53      0.53     23037

 Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1.0,1,0.0,1.0
1,1.0,1,0.0,1.0
2,1.0,0,1.0,0.0
3,0.0,1,0.0,1.0
4,1.0,1,0.0,1.0



 Evaluando modelo: unsw15_logistic_regression_model_unsw
 Probabilidades obtenidas (modelo original).
 F1-score (weighted): 0.5460   Accuracy: 0.5527
 Classification Report:
              precision    recall  f1-score   support

         0.0       0.36      0.32      0.34      8314
         1.0       0.64      0.68      0.66     14723

    accuracy                           0.55     23037
   macro avg       0.50      0.50      0.50     23037
weighted avg       0.54      0.55      0.55     23037

 Primeras predicciones:


...

,y_true,y_pred,proba_clase_0,proba_clase_1
0,1.0,1,0.000000,1.000000e+00
1,1.0,1,0.000000,1.000000e+00
2,1.0,0,1.000000,1.637427e-07
3,0.0,1,0.000000,1.000000e+00
4,1.0,1,0.260469,7.395307e-01



 Evaluando modelo: unsw15_knn_model_unsw
 Probabilidades obtenidas (modelo original).
 F1-score (weighted): 0.5364   Accuracy: 0.5340
 Classification Report:
              precision    recall  f1-score   support

         0.0       0.36      0.38      0.37      8314
         1.0       0.64      0.62      0.63     14723

    accuracy                           0.53     23037
   macro avg       0.50      0.50      0.50     23037
weighted avg       0.54      0.53      0.54     23037

 Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1.0,1,0.000000,1.000000
1,1.0,1,0.000000,1.000000
2,1.0,0,1.000000,0.000000
3,0.0,1,0.000000,1.000000
4,1.0,1,0.437518,0.562482



 Evaluando modelo: unsw15_rf_stacking_model_optimized_unsw
 Stacking: probabilidades obtenidas.
 F1-score (weighted): 0.5316   Accuracy: 0.5255
 Classification Report:
              precision    recall  f1-score   support

         0.0       0.36      0.42      0.39      8314
         1.0       0.64      0.59      0.61     14723

    accuracy                           0.53     23037
   macro avg       0.50      0.50      0.50     23037
weighted avg       0.54      0.53      0.53     23037

 Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1.0,1,0.000000,1.000000
1,1.0,1,0.000000,1.000000
2,1.0,0,1.000000,0.000000
3,0.0,1,0.000000,1.000000
4,1.0,1,0.037479,0.962521


#### CICIDS2018

In [ ]:
from sklearn.metrics            import accuracy_score, f1_score, classification_report
from sklearn.calibration       import CalibratedClassifierCV
from IPython.display           import display
import pandas as pd
import numpy as np

# ——————————————
# 1) Datos binarios
#    (X_test_bin, y_test_bin ya separados) - > LITO
# ——————————————

# ——————————————
# 2) Filtra solo los modelos binarios (ej. CICIDS2018)
# ——————————————
binary_models = {
    name: mdl
    for name, mdl in models.items()
    if name.lower().startswith("cicids2018_")
}

print("=== Evaluación BINARIA (Threat) ===")
for model_name, model in binary_models.items():
    print(f"\n Evaluando (binario) modelo: {model_name}")

    # ————— Evaluación directa  —————
    m = model
    print(" Usando el modelo original sobre el conjunto de prueba.")

    has_proba = False

    # ————— Prob. KERAS —————
    if isinstance(model, KerasModel):
        y_pred, y_proba = keras_pred_and_proba_ids(model, X_test_bin)
        has_proba = True
        print(" Keras salida.")

    # ——— Resto de modelos (la chusma) ———
    else:
        m = model
        print(" Usando el modelo original sobre el conjunto de prueba.")

        # Predicción de etiquetas
        raw_pred = m.predict(X_test_bin)
        raw_pred = np.asarray(raw_pred)

        # Si devolvió floats continuos, binarizamos por 0.5
        if raw_pred.dtype.kind == 'f':
            y_pred = (raw_pred.flatten() >= 0.5).astype(int)
        else:
            y_pred = raw_pred

        # Intento de probabilidades sklearn
        try:
            y_proba = m.predict_proba(X_test_bin)
            has_proba = True
            print(" Probabilidades sklearn OK.")
        except Exception:
            y_proba = None
            has_proba = False
            print(" No hay predict_proba disponible.")


    # ————— Métricas —————
    acc = accuracy_score(y_test_bin, y_pred)
    f1  = f1_score    (y_test_bin, y_pred, average="weighted")
    print(f" Accuracy: {acc:.4f} — F1 (w): {f1:.4f}")
    print(" Classification Report:")
    print(classification_report(y_test_bin, y_pred, zero_division=0))

    # ————— Primeras predicciones —————
    df_out = pd.DataFrame({"true": y_test_bin, "pred": y_pred})
    if has_proba:
        proba_df = pd.DataFrame(
            y_proba,
            columns=[f"proba_clase_{i}" for i in range(y_proba.shape[1])]
        )
        df_out = pd.concat([df_out.reset_index(drop=True), proba_df], axis=1)

    print(" Primeras predicciones:")
    display(df_out.head())


=== Evaluación BINARIA (Threat) ===

 Evaluando (binario) modelo: cicids2018_xgb_model_bin_cicids2018_cicids2018
 Usando el modelo original sobre el conjunto de prueba.
 Usando el modelo original sobre el conjunto de prueba.
 Probabilidades sklearn OK.
 Accuracy: 0.9982 — F1 (w): 0.9982
 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    239505
           1       0.99      1.00      0.99     31080

    accuracy                           1.00    270585
   macro avg       0.99      1.00      1.00    270585
weighted avg       1.00      1.00      1.00    270585

 Primeras predicciones:


,true,pred,proba_clase_0,proba_clase_1
0,0,0,0.999438,0.000562
1,0,0,0.998856,0.001144
2,0,0,0.999307,0.000693
3,0,0,0.998580,0.001420
4,0,0,0.997001,0.002999



 Evaluando (binario) modelo: cicids2018_lightgbm_bin_cicids2018_cicids2018
 Usando el modelo original sobre el conjunto de prueba.
 Usando el modelo original sobre el conjunto de prueba.
[LightGBM] [Warning] feature_fraction is set=0.8450178757972995, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8450178757972995
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] bagging_fraction is set=0.5222276663053877, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5222276663053877
[LightGBM] [Warning] feature_fraction is set=0.8450178757972995, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8450178757972995
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] bagging_fraction is set=0.5222276663053877, subsample=1.0 will be ignored. Current value: bagging_fraction=0.52222766630

,true,pred,proba_clase_0,proba_clase_1
0,0,0,0.999999,1.483174e-06
1,0,0,0.999998,1.990587e-06
2,0,0,0.999999,6.102360e-07
3,0,0,0.999997,3.293257e-06
4,0,0,0.999999,1.438984e-06



 Evaluando (binario) modelo: cicids2018_decisiontree_bin_cicids2018_cicids2018
 Usando el modelo original sobre el conjunto de prueba.
 Usando el modelo original sobre el conjunto de prueba.
 Probabilidades sklearn OK.
 Accuracy: 0.9999 — F1 (w): 0.9999
 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    239505
           1       1.00      1.00      1.00     31080

    accuracy                           1.00    270585
   macro avg       1.00      1.00      1.00    270585
weighted avg       1.00      1.00      1.00    270585

 Primeras predicciones:


...

,true,pred,proba_clase_0,proba_clase_1
0,0,0,1.000000e+00,9.611954e-72
1,0,1,2.286296e-14,1.000000e+00
2,0,0,1.000000e+00,2.065292e-248
3,0,0,1.000000e+00,5.031432e-18
4,0,0,1.000000e+00,5.062567e-12



 Evaluando (binario) modelo: cicids2018_catboost_bin_2018_cicids2018
 Usando el modelo original sobre el conjunto de prueba.
 Usando el modelo original sobre el conjunto de prueba.
 Probabilidades sklearn OK.
 Accuracy: 1.0000 — F1 (w): 1.0000
 Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    239505
           1       1.00      1.00      1.00     31080

    accuracy                           1.00    270585
   macro avg       1.00      1.00      1.00    270585
weighted avg       1.00      1.00      1.00    270585

 Primeras predicciones:


,true,pred,proba_clase_0,proba_clase_1
0,0,0,1.00000,2.107855e-08
1,0,0,0.99997,2.960083e-05
2,0,0,1.00000,3.913196e-07
3,0,0,1.00000,5.517997e-08
4,0,0,1.00000,8.561738e-09



 Evaluando (binario) modelo: cicids2018_simple_neural_network_model_bin_cicids2018
 Usando el modelo original sobre el conjunto de prueba.
 Keras salida.
 Accuracy: 0.8851 — F1 (w): 0.8312
 Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94    239505
           1       0.00      0.00      0.00     31080

    accuracy                           0.89    270585
   macro avg       0.44      0.50      0.47    270585
weighted avg       0.78      0.89      0.83    270585

 Primeras predicciones:


,true,pred,proba_clase_0,proba_clase_1
0,0,0,0.884408,0.115592
1,0,0,0.884408,0.115592
2,0,0,0.884408,0.115592
3,0,0,0.884408,0.115592
4,0,0,0.884408,0.115592


# Calivra

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model as KerasModel
from IPython.display import display
import numpy as np
import pandas as pd

# ============================================================
# Sección única: evaluación con calibración para los datasets
# ============================================================

def _get_pred_proba_keras(model, X):
  return keras_pred_and_proba_ids(model, X)


def _calibrate_sklearn_model(model, X_calib, y_calib, desc=""):
    """
    Envuelve un modelo sklearn en CalibratedClassifierCV(cv='prefit'),
    usando solo datos de calibración (sin test).
    Si algo falla, devuelve el modelo original.
    """
    try:
        cal = CalibratedClassifierCV(base_estimator=model, cv="prefit", method="sigmoid")
        cal.fit(X_calib, y_calib)
        print(f"   -> {desc}: calibrado con éxito sobre el split de calibración.")
        return cal
    except Exception as e:
        print(f"   -> {desc}: NO se pudo calibrar ({type(e).__name__}), se usa el modelo original.")
        return model


def _eval_model(model, X_eval, y_eval, nombre_modelo="", dataset_name=""):
    """
    Calcula métricas y devuelve (metrics_dict, df_predicciones).
    Usa F1 ponderado (weighted) y precision/recall binarios con la clase 1 como positiva.
    """
    has_proba = False
    y_proba = None

    # Keras
    if isinstance(model, KerasModel):
        try:
            y_pred, y_proba = _get_pred_proba_keras(model, X_eval)
            has_proba = True
        except Exception:
            y_pred = model.predict(X_eval)
            y_pred = np.asarray(y_pred).reshape(-1).astype(int)
    else:
        # sklearn / otros
        y_pred = model.predict(X_eval)
        try:
            y_proba = model.predict_proba(X_eval)
            has_proba = True
        except Exception:
            y_proba = None
            has_proba = False

    # Métricas principales
    acc = accuracy_score(y_eval, y_pred)
    f1_w = f1_score(y_eval, y_pred, average="weighted", zero_division=0)
    prec_pos = precision_score(y_eval, y_pred, zero_division=0)
    rec_pos = recall_score(y_eval, y_pred, zero_division=0)

    roc = None
    if has_proba and y_proba is not None and y_proba.ndim == 2 and y_proba.shape[1] == 2:
        try:
            roc = roc_auc_score(y_eval, y_proba[:, 1])
        except Exception:
            roc = None

    print(f"   Accuracy: {acc:.4f} | F1 (weighted): {f1_w:.4f} | Precision(+): {prec_pos:.4f} | Recall(+): {rec_pos:.4f}")
    if roc is not None:
        print(f"   ROC AUC: {roc:.4f}")
    print("   Classification report:")
    print(classification_report(y_eval, y_pred, zero_division=0))

    # DataFrame con primeras predicciones
    df_out = pd.DataFrame({"y_true": y_eval, "y_pred": y_pred})
    if has_proba and y_proba is not None:
        proba_cols = [f"proba_clase_{i}" for i in range(y_proba.shape[1])]
        proba_df = pd.DataFrame(y_proba, columns=proba_cols)
        df_out = pd.concat([df_out.reset_index(drop=True), proba_df], axis=1)

    metrics = {
        "dataset": dataset_name,
        "modelo": nombre_modelo,
        "accuracy": acc,
        "f1_weighted": f1_w,
        "precision_pos": prec_pos,
        "recall_pos": rec_pos,
        "roc_auc": roc,
    }
    return metrics, df_out


# ------------------------------------------------------------------
# A) NSL-KDD: usar X_train_scaled_df como calibración, X_test_scaled_df como test
# ------------------------------------------------------------------
print("\n================ NSL-KDD — evaluación con calibración ================")

# Split de calibración (train)
X_nsl_calib = X_train_scaled_df.drop("attack", axis=1)
y_nsl_calib = X_train_scaled_df["attack"]

# mask_calib = y_nsl_calib.notna()
# X_nsl_calib = X_nsl_calib.loc[mask_calib]
# y_nsl_calib = y_nsl_calib.loc[mask_calib]

# Split de test
X_nsl_test = X_test_scaled_df.drop("attack", axis=1)
y_nsl_test = X_test_scaled_df["attack"]

# mask_test = y_nsl_test.notna()
# X_nsl_test = X_nsl_test.loc[mask_test]
# y_nsl_test = y_nsl_test.loc[mask_test]

nslkdd_models = {
    name: mdl
    for name, mdl in models.items()
    if name.lower().startswith("nslkdd_")
}

base_keys_nsl = [
    "nslkdd_decision_tree_model",
    "nslkdd_logistic_regression_model",
    "nslkdd_svc_best_clf_model",
    "nslkdd_knn_model",
]

metrics_nsl_list = []

for model_name, model in nslkdd_models.items():
    print(f"\n[NSL-KDD] Modelo: {model_name}")

    # Stacking RF
    if "rf_stacking_model_optimized" in model_name:
        # Construir features apilados tanto para calib como para test
        base_preds_calib = [
            nslkdd_models[bk].predict(X_nsl_calib) for bk in base_keys_nsl
        ]
        base_preds_test = [
            nslkdd_models[bk].predict(X_nsl_test) for bk in base_keys_nsl
        ]
        X_calib_stack = np.hstack([X_nsl_calib.values] + [p.reshape(-1, 1) for p in base_preds_calib])
        X_test_stack = np.hstack([X_nsl_test.values] + [p.reshape(-1, 1) for p in base_preds_test])

        model_to_use = _calibrate_sklearn_model(
            model,
            X_calib_stack,
            y_nsl_calib,
            desc=f"{model_name} (stacking NSL-KDD)",
        )
        metrics, df_out = _eval_model(
            model_to_use,
            X_test_stack,
            y_nsl_test,
            nombre_modelo=model_name,
            dataset_name="nslkdd",
        )
    # Modelos Keras
    elif isinstance(model, KerasModel):
        print("   (Modelo Keras: se evalúa sin CalibratedClassifierCV, solo con sus probabilidades).")
        metrics, df_out = _eval_model(
            model,
            X_nsl_test,
            y_nsl_test,
            nombre_modelo=model_name,
            dataset_name="nslkdd",
        )
    # Resto de modelos sklearn
    else:
        model_to_use = _calibrate_sklearn_model(
            model,
            X_nsl_calib,
            y_nsl_calib,
            desc=f"{model_name} (NSL-KDD)",
        )
        metrics, df_out = _eval_model(
            model_to_use,
            X_nsl_test,
            y_nsl_test,
            nombre_modelo=model_name,
            dataset_name="nslkdd",
        )

    metrics_nsl_list.append(metrics)
    print("   Primeras predicciones:")
    display(df_out.head())

metrics_nsl = pd.DataFrame(metrics_nsl_list).set_index("modelo")
print("\n Resumen métricas NSL-KDD (con calibración):")
display(metrics_nsl)

# ------------------------------------------------------------------
# B) UNSW15: split local en train/calib/test usando datasets["unsw15"]
# ------------------------------------------------------------------
print("\n================ UNSW15 — evaluación CON calibración ================")

unsw_df = datasets["unsw15"]
X_unsw_all = unsw_df.drop("attack", axis=1)
y_unsw_all = unsw_df["attack"]

mask_unsw = y_unsw_all.notna()
X_unsw_all = X_unsw_all.loc[mask_unsw]
y_unsw_all = y_unsw_all.loc[mask_unsw]

# Split: 70% train, 15% calib, 15% test (aprox)
X_unsw_train, X_unsw_tmp, y_unsw_train, y_unsw_tmp = train_test_split(
    X_unsw_all, y_unsw_all,
    test_size=0.30,
    random_state=57,
    stratify=y_unsw_all,
)
X_unsw_calib, X_unsw_test, y_unsw_calib, y_unsw_test = train_test_split(
    X_unsw_tmp, y_unsw_tmp,
    test_size=0.50,
    random_state=57,
    stratify=y_unsw_tmp,
)

unsw_models = {
    name: mdl
    for name, mdl in models.items()
    if name.lower().startswith("unsw15_")
}

base_keys_unsw = [
    "unsw15_decision_tree_model_unsw",
    "unsw15_logistic_regression_model_unsw",
    "unsw15_svc_model_unsw",
    "unsw15_knn_model_unsw",
]

metrics_unsw_list = []

for model_name, model in unsw_models.items():
    print(f"\n[UNSW15] Modelo: {model_name}")

    # Stacking RF
    if "rf_stacking_model_optimized_unsw" in model_name:
        base_preds_calib = [
            unsw_models[bk].predict(X_unsw_calib) for bk in base_keys_unsw
        ]
        base_preds_test = [
            unsw_models[bk].predict(X_unsw_test) for bk in base_keys_unsw
        ]
        X_calib_stack = np.hstack([X_unsw_calib.values] + [p.reshape(-1, 1) for p in base_preds_calib])
        X_test_stack = np.hstack([X_unsw_test.values] + [p.reshape(-1, 1) for p in base_preds_test])

        model_to_use = _calibrate_sklearn_model(
            model,
            X_calib_stack,
            y_unsw_calib,
            desc=f"{model_name} (stacking UNSW15)",
        )
        metrics, df_out = _eval_model(
            model_to_use,
            X_test_stack,
            y_unsw_test,
            nombre_modelo=model_name,
            dataset_name="unsw15",
        )
    elif isinstance(model, KerasModel):
        print("   (Modelo Keras: se evalúa sin CalibratedClassifierCV).")
        metrics, df_out = _eval_model(
            model,
            X_unsw_test,
            y_unsw_test,
            nombre_modelo=model_name,
            dataset_name="unsw15",
        )
    else:
        model_to_use = _calibrate_sklearn_model(
            model,
            X_unsw_calib,
            y_unsw_calib,
            desc=f"{model_name} (UNSW15)",
        )
        metrics, df_out = _eval_model(
            model_to_use,
            X_unsw_test,
            y_unsw_test,
            nombre_modelo=model_name,
            dataset_name="unsw15",
        )

    metrics_unsw_list.append(metrics)
    print("   Primeras predicciones:")
    display(df_out.head())

metrics_unsw = pd.DataFrame(metrics_unsw_list).set_index("modelo")
print("\n Resumen métricas UNSW15 (con calibración):")
display(metrics_unsw)

# ------------------------------------------------------------------
# C) CICIDS2018 (binario Threat) usando X_train_bin / X_test_bin
# ------------------------------------------------------------------
print("\n================ CICIDS2018 (binario) — evaluación CON calibración ================")

binary_models = {
    name: mdl
    for name, mdl in models.items()
    if name.lower().startswith("cicids2018_")
}

metrics_cic_list = []

for model_name, model in binary_models.items():
    print(f"\n[CICIDS2018-bin] Modelo: {model_name}")

    if isinstance(model, KerasModel):
        print("   (Modelo Keras: se evalúa sin CalibratedClassifierCV).")
        metrics, df_out = _eval_model(
            model,
            X_test_bin,
            y_test_bin,
            nombre_modelo=model_name,
            dataset_name="cicids2018_bin",
        )
    else:
        # Calibrar con el train binario y evaluar en el test binario
        model_to_use = _calibrate_sklearn_model(
            model,
            X_train_bin,
            y_train_bin,
            desc=f"{model_name} (CICIDS2018 binario)",
        )
        metrics, df_out = _eval_model(
            model_to_use,
            X_test_bin,
            y_test_bin,
            nombre_modelo=model_name,
            dataset_name="cicids2018_bin",
        )

    metrics_cic_list.append(metrics)
    print("   Primeras predicciones:")
    display(df_out.head())

metrics_cic = pd.DataFrame(metrics_cic_list).set_index("modelo")
print("\n Resumen métricas CICIDS2018 binario (con calibración):")
display(metrics_cic)



================ NSL-KDD — evaluación CON calibración ================

[NSL-KDD] Modelo: nslkdd_xgb_model
   -> nslkdd_xgb_model (NSL-KDD): NO se pudo calibrar (TypeError), se usa el modelo original.
   Accuracy: 0.9965 | F1 (weighted): 0.9965 | Precision(+): 0.9963 | Recall(+): 0.9972
   ROC AUC: 0.9998
   Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5828
           1       1.00      1.00      1.00      6770

    accuracy                           1.00     12598
   macro avg       1.00      1.00      1.00     12598
weighted avg       1.00      1.00      1.00     12598

   Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,1,1,0.001642,0.998358
1,1,1,0.017005,0.982995
2,0,0,0.997723,0.002278
3,1,1,0.001487,0.998513
4,0,0,0.994979,0.005021



[NSL-KDD] Modelo: nslkdd_svc_best_clf_model
   -> nslkdd_svc_best_clf_model (NSL-KDD): NO se pudo calibrar (TypeError), se usa el modelo original.
   Accuracy: 0.9834 | F1 (weighted): 0.9834 | Precision(+): 0.9821 | Recall(+): 0.9871
   Classification report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      5828
           1       0.98      0.99      0.98      6770

    accuracy                           0.98     12598
   macro avg       0.98      0.98      0.98     12598
weighted avg       0.98      0.98      0.98     12598

   Primeras predicciones:


,y_true,y_pred
0,1,1
1,1,1
2,0,0
3,1,1
4,0,0



[NSL-KDD] Modelo: nslkdd_gnb_model
   -> nslkdd_gnb_model (NSL-KDD): NO se pudo calibrar (TypeError), se usa el modelo original.
   Accuracy: 0.8919 | F1 (weighted): 0.8914 | Precision(+): 0.8740 | Recall(+): 0.9334
   ROC AUC: 0.9425
   Classification report:
              precision    recall  f1-score   support

           0       0.92      0.84      0.88      5828
           1       0.87      0.93      0.90      6770

    accuracy                           0.89     12598
   macro avg       0.89      0.89      0.89     12598
weighted avg       0.89      0.89      0.89     12598

   Primeras predicciones:


...

,y_true,y_pred,proba_clase_0,proba_clase_1
0,0,0,1.00000,2.107855e-08
1,0,0,0.99997,2.960083e-05
2,0,0,1.00000,3.913196e-07
3,0,0,1.00000,5.517997e-08
4,0,0,1.00000,8.561738e-09



[CICIDS2018-bin] Modelo: cicids2018_simple_neural_network_model_bin_cicids2018
   (Modelo Keras: se evalúa sin CalibratedClassifierCV).
   Accuracy: 0.8851 | F1 (weighted): 0.8312 | Precision(+): 0.0000 | Recall(+): 0.0000
   ROC AUC: 0.5000
   Classification report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94    239505
           1       0.00      0.00      0.00     31080

    accuracy                           0.89    270585
   macro avg       0.44      0.50      0.47    270585
weighted avg       0.78      0.89      0.83    270585

   Primeras predicciones:


,y_true,y_pred,proba_clase_0,proba_clase_1
0,0,0,0.884408,0.115592
1,0,0,0.884408,0.115592
2,0,0,0.884408,0.115592
3,0,0,0.884408,0.115592
4,0,0,0.884408,0.115592



 Resumen métricas CICIDS2018 binario (con calibración):


,dataset,accuracy,f1_weighted,precision_pos,recall_pos,roc_auc
modelo,,,,,,
cicids2018_xgb_model_bin_cicids2018_cicids2018,cicids2018_bin,0.998208,0.998213,0.985311,0.999292,0.999982
cicids2018_lightgbm_bin_cicids2018_cicids2018,cicids2018_bin,0.999963,0.999963,0.999903,0.999775,0.999997
cicids2018_decisiontree_bin_cicids2018_cicids2018,cicids2018_bin,0.999897,0.999897,0.999421,0.999678,0.999808
cicids2018_final_gnb_bin_cicids2018_cicids2018,cicids2018_bin,0.830915,0.857564,0.403119,0.982143,0.954903
cicids2018_catboost_bin_2018_cicids2018,cicids2018_bin,0.999959,0.999959,0.999839,0.999807,0.999995
cicids2018_simple_neural_network_model_bin_cicids2018,cicids2018_bin,0.885138,0.831206,0.000000,0.000000,0.500000
